# AIC 2026 — Textual KIS Retrieval Pipeline (Colab Pro)

Notebook này triển khai pipeline Textual Known Item Search dựa trên **keyframes có sẵn** và `map-keyframes/`, không trích xuất lại frame từ video.

Pipeline chính:

1. Build `master_keyframes.parquet` từ `keyframes/ + map-keyframes/`.
2. Chuẩn hóa object JSON hiện có vào `objects/objects_index.parquet` và `objects/keyframe_object_summary.parquet`.
3. Tạo visual embeddings bằng **Qwen3-VL-Embedding-2B** và FAISS index.
4. Multi-query retrieval: query gốc + bản English/query-concepts.
5. Base-object scoring từ Faster R-CNN JSON.
6. Metadata BM25 từ `media-info/`.
7. **Grounding DINO** query-time verification trên top candidates để kiểm tra object/attribute open-vocabulary.
8. OCR lazy khi query chứa tên/logo/chữ/số cụ thể.
9. Temporal grouping bằng `pts_time` của keyframes.
10. **Qwen3-VL-Reranker-2B** rerank top segments bằng 3 keyframes lân cận.
11. Diversity-aware Top-100 và map keyframe cuối cùng về `frame_idx` thật.

Cấu trúc dữ liệu giả định:

```text
/content/drive/MyDrive/data/
├── video/
├── keyframes/
│   ├── L01_V001/
│   │   ├── 000.jpg
│   │   ├── 001.jpg
│   │   └── ...
├── map-keyframes/
│   ├── L01_V001.csv
│   └── ...
├── media-info/
└── objects/
```

Notebook **không xóa** object JSON gốc. Nó chỉ tạo thêm index/cache trong `objects/`.

In [1]:
# ============================================================
# 0. INSTALL
# ============================================================
# Không ép cài lại torch để tránh phá CUDA stack của Colab.

!pip -q install -U \
    "transformers>=4.57.3" \
    "accelerate>=1.12.0" \
    "qwen-vl-utils>=0.0.14" \
    huggingface_hub \
    faiss-cpu \
    pyarrow \
    rank-bm25 \
    rapidfuzz \
    unidecode \
    easyocr \
    sentencepiece

!rm -rf /content/Qwen3-VL-Embedding
!git clone -q --depth 1 https://github.com/QwenLM/Qwen3-VL-Embedding.git /content/Qwen3-VL-Embedding

print("Dependencies installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 111.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 793.2/793.2 kB 52.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 111.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 52.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 123.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 98.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 73.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 63.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.2/296.2 kB 29.4 MB/s eta 0:00:00
Dependencies installed.


In [3]:
# ============================================================
# 1. MOUNT DRIVE + CONFIG
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, sys, json, math, hashlib, gc, re, warnings
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from PIL import Image

ROOT = Path('/content/drive/MyDrive/AI Challenge/data')

# RAW DATA DIRECTORIES
VIDEO_DIR = ROOT / "video"
KEYFRAME_DIR = ROOT / "keyframes"
MAP_DIR = ROOT / "map-keyframes"
MEDIA_DIR = ROOT / "media-info"

# Raw object JSON của BTC.
# Chỉ đọc, KHÔNG ghi dữ liệu processed vào đây.
OBJECTS_DIR = ROOT / "objects"

# PROCESSED OBJECT DATA
PROCESSED_OBJECTS_DIR = ROOT / "processed-objects"
PROCESSED_OBJECTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)
GROUNDING_CACHE_DIR = (
    PROCESSED_OBJECTS_DIR / "grounding-cache"
)
GROUNDING_CACHE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# OTHER RETRIEVAL ARTIFACTS
ARTIFACT_DIR = ROOT / "retrieval-artifacts"
ARTIFACT_DIR.mkdir(
    parents=True,
    exist_ok=True
)
OCR_CACHE_DIR = (
    ARTIFACT_DIR / "ocr-cache"
)
OCR_CACHE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# INDEX PATHS
MASTER_PATH = (
    ARTIFACT_DIR / "master_keyframes.parquet"
)
MEDIA_INDEX_PATH = (
    ARTIFACT_DIR / "media_index.parquet"
)
BASE_OBJECT_INDEX_PATH = (
    PROCESSED_OBJECTS_DIR /
    "objects_index.parquet"
)
BASE_OBJECT_SUMMARY_PATH = (
    PROCESSED_OBJECTS_DIR /
    "keyframe_object_summary.parquet"
)

QWEN_MODEL_ID = 'Qwen/Qwen3-VL-Embedding-2B'
QWEN_RERANKER_ID = 'Qwen/Qwen3-VL-Reranker-2B'
GROUNDING_DINO_ID = 'IDEA-Research/grounding-dino-tiny'
PARSER_MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'

# Qwen3-VL-Embedding-2B: full dim 2048. Dùng prefix 1024 để giảm RAM/index size.
INDEX_DIM = 1024
QWEN_MIN_PIXELS = 32 * 32
QWEN_MAX_PIXELS = 768 * 768

QWEN_EMBED_PATH = ARTIFACT_DIR / f'qwen3_vl_embedding_{INDEX_DIM}d.npy'
QWEN_PROGRESS_PATH = ARTIFACT_DIR / f'qwen3_vl_embedding_{INDEX_DIM}d.progress.json'
FAISS_PATH = ARTIFACT_DIR / f'qwen3_vl_{INDEX_DIM}d.index'

TOPK_PER_QUERY = 1200
TOPK_AFTER_RRF = 1200
EXPENSIVE_VERIFY_TOPK = 200
RERANK_TOP_SEGMENTS = 40
TEMPORAL_GAP_SECONDS = 12.0
BASE_OBJECT_MIN_CONF = 0.20

for p in [VIDEO_DIR, KEYFRAME_DIR, MAP_DIR, MEDIA_DIR, OBJECTS_DIR]:
    print(f'{p}:', 'OK' if p.exists() else 'MISSING')

assert KEYFRAME_DIR.exists(), KEYFRAME_DIR
assert MAP_DIR.exists(), MAP_DIR

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/AI Challenge/data/video: OK
/content/drive/MyDrive/AI Challenge/data/keyframes: OK
/content/drive/MyDrive/AI Challenge/data/map-keyframes: OK
/content/drive/MyDrive/AI Challenge/data/media-info: OK
/content/drive/MyDrive/AI Challenge/data/objects: OK


In [4]:
# ============================================================
# 2. RUNTIME CHECK
# ============================================================
print('torch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())

if torch.cuda.is_available():
    prop = torch.cuda.get_device_properties(0)
    vram_gb = prop.total_memory / 1024**3
    print('GPU:', prop.name)
    print(f'VRAM: {vram_gb:.1f} GB')
    print('BF16 supported:', torch.cuda.is_bf16_supported())
else:
    vram_gb = 0.0
    warnings.warn('GPU not found; preprocessing will be slow.')

DTYPE = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
QWEN_BATCH_SIZE = 8 if vram_gb >= 35 else 4 if vram_gb >= 20 else 2
print('dtype:', DTYPE)
print('Qwen batch:', QWEN_BATCH_SIZE)

torch: 2.11.0+cu128
CUDA: True
GPU: NVIDIA A100-SXM4-40GB
VRAM: 39.5 GB
BF16 supported: True
dtype: torch.bfloat16
Qwen batch: 8


In [5]:
MASTER_CSV_PATH = ROOT / "keyframe_master.csv"

assert MASTER_CSV_PATH.exists(), (
    f"Không tìm thấy {MASTER_CSV_PATH}"
)

master = pd.read_csv(
    MASTER_CSV_PATH,
    dtype={
        "video_id": "string",
        "keyframe_id": "string",
    }
)

master = (
    master
    .sort_values("global_id")
    .reset_index(drop=True)
)

print(f"Master rows : {len(master):,}")
print(f"Videos      : {master['video_id'].nunique():,}")

display(master.head())

Master rows : 177,321
Videos      : 873


,Unnamed: 0,global_id,video_id,keyframe_id,n,pts_time,fps,frame_idx,image_path,map_mode
0,0,0,L21_V001,001,1,0.0000,30.0,0,/content/drive/MyDrive/data/keyframes/L21_V001...,direct_n
1,1,1,L21_V001,002,2,3.0000,30.0,90,/content/drive/MyDrive/data/keyframes/L21_V001...,direct_n
2,2,2,L21_V001,003,3,8.7000,30.0,261,/content/drive/MyDrive/data/keyframes/L21_V001...,direct_n
3,3,3,L21_V001,004,4,11.7333,30.0,351,/content/drive/MyDrive/data/keyframes/L21_V001...,direct_n
4,4,4,L21_V001,005,5,13.7000,30.0,411,/content/drive/MyDrive/data/keyframes/L21_V001...,direct_n


In [ ]:
print(
    master[
        ["video_id", "keyframe_id", "frame_idx"]
    ].head(10)
)

In [ ]:
# ============================================================
# PARALLEL OBJECT NORMALIZATION
# ============================================================

NUM_SESSIONS = 2

# ============================================================
# SESSION 1:
# SESSION_ID = 0
#
# SESSION 2:
# SESSION_ID = 1
# ============================================================

SESSION_ID = 0

assert 0 <= SESSION_ID < NUM_SESSIONS

print(
    f"Running object normalization "
    f"session {SESSION_ID + 1}/{NUM_SESSIONS}"
)

Running object normalization session 1/2


In [ ]:
OBJECTS_DIR = ROOT / "objects"

PROCESSED_OBJECTS_DIR = (
    ROOT / "processed-objects"
)

PROCESSED_OBJECTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

OBJECT_SHARD_DIR = (
    PROCESSED_OBJECTS_DIR / "shards"
)

OBJECT_SHARD_DIR.mkdir(
    parents=True,
    exist_ok=True
)


BASE_OBJECT_INDEX_PATH = (
    PROCESSED_OBJECTS_DIR /
    "objects_index.parquet"
)

BASE_OBJECT_SUMMARY_PATH = (
    PROCESSED_OBJECTS_DIR /
    "keyframe_object_summary.parquet"
)

In [ ]:
def get_session_master(
    master_df,
    session_id,
    num_sessions=2
):
    master_df = (
        master_df
        .sort_values("global_id")
        .reset_index(drop=True)
    )

    boundaries = np.linspace(
        0,
        len(master_df),
        num_sessions + 1,
        dtype=int
    )

    start = boundaries[session_id]
    end = boundaries[session_id + 1]

    shard = (
        master_df
        .iloc[start:end]
        .copy()
        .reset_index(drop=True)
    )

    print(
        f"Session {session_id}: "
        f"[{start:,}, {end:,})"
    )

    print(
        f"Number of keyframes: "
        f"{len(shard):,}"
    )

    if len(shard) > 0:
        print(
            "global_id:",
            shard["global_id"].min(),
            "->",
            shard["global_id"].max()
        )

    return shard

In [ ]:
session_master = get_session_master(
    master,
    SESSION_ID,
    NUM_SESSIONS
)

Session 0: [0, 88,660)
Number of keyframes: 88,660
global_id: 0 -> 88659


In [ ]:
SESSION_OBJECT_INDEX_PATH = (
    OBJECT_SHARD_DIR /
    (
        f"objects_index."
        f"part-{SESSION_ID:02d}"
        f"-of-{NUM_SESSIONS:02d}.parquet"
    )
)

SESSION_OBJECT_SUMMARY_PATH = (
    OBJECT_SHARD_DIR /
    (
        f"keyframe_object_summary."
        f"part-{SESSION_ID:02d}"
        f"-of-{NUM_SESSIONS:02d}.parquet"
    )
)

SESSION_DONE_PATH = (
    OBJECT_SHARD_DIR /
    (
        f"done."
        f"part-{SESSION_ID:02d}"
        f"-of-{NUM_SESSIONS:02d}.json"
    )
)

## 3. Master keyframe index

Mỗi keyframe được map chính xác sang `frame_idx` thật bằng file CSV tương ứng.

Code ưu tiên `int(keyframe_stem) == n`. Nếu tên ảnh và `n` lệch offset nhưng số lượng bằng nhau, code fallback sang ghép theo sorted order và báo warning.

In [ ]:
# ============================================================
# 3. BUILD MASTER KEYFRAME INDEX
# ============================================================
REQUIRED_MAP_COLS = {'n', 'pts_time', 'fps', 'frame_idx'}

def numeric_stem(path):
    try:
        return int(path.stem)
    except Exception:
        return None

def build_master_index(force=False):
    if MASTER_PATH.exists() and not force:
        df = pd.read_parquet(MASTER_PATH)
        print(f'Loaded master: {len(df):,} rows')
        return df

    rows, warns = [], []
    gid = 0
    video_folders = sorted([p for p in KEYFRAME_DIR.iterdir() if p.is_dir()])

    for vdir in tqdm(video_folders, desc='Videos'):
        video_id = vdir.name
        csv_path = MAP_DIR / f'{video_id}.csv'
        if not csv_path.exists():
            warns.append(f'[NO MAP] {video_id}')
            continue

        m = pd.read_csv(csv_path)
        missing = REQUIRED_MAP_COLS - set(m.columns)
        if missing:
            raise ValueError(f'{csv_path} missing {missing}')
        m = m.sort_values('n').reset_index(drop=True)

        imgs = list(vdir.glob('*.jpg')) + list(vdir.glob('*.jpeg')) + list(vdir.glob('*.png'))
        imgs = sorted(imgs, key=lambda p: (numeric_stem(p) is None, numeric_stem(p) if numeric_stem(p) is not None else p.stem))
        if not imgs:
            warns.append(f'[NO IMAGES] {video_id}')
            continue

        n_to_row = {int(r['n']): r for _, r in m.iterrows()}
        direct_ok = all(numeric_stem(img) is not None and numeric_stem(img) in n_to_row for img in imgs)

        if direct_ok:
            pairs = [(img, n_to_row[numeric_stem(img)]) for img in imgs]
            mode = 'direct_n'
        elif len(imgs) == len(m):
            warns.append(f'[ORDER FALLBACK] {video_id}')
            pairs = list(zip(imgs, [r for _, r in m.iterrows()]))
            mode = 'sorted_order'
        else:
            raise ValueError(f'{video_id}: keyframes={len(imgs)}, map rows={len(m)}; cannot align')

        for img, r in pairs:
            rows.append({
                'global_id': gid,
                'video_id': video_id,
                'keyframe_id': img.stem,
                'n': int(r['n']),
                'pts_time': float(r['pts_time']),
                'fps': float(r['fps']),
                'frame_idx': int(r['frame_idx']),
                'image_path': str(img),
                'map_mode': mode,
            })
            gid += 1

    df = pd.DataFrame(rows).sort_values('global_id').reset_index(drop=True)
    df.to_parquet(MASTER_PATH, index=False)
    print(f'Saved {len(df):,} keyframes; videos={df.video_id.nunique():,}')
    for w in warns[:30]: print(' -', w)
    if len(warns) > 30: print(' ...', len(warns)-30, 'more')
    return df

master = build_master_index(force=False)
display(master.head())

Videos:   0%|          | 0/873 [00:00<?, ?it/s]

Saved 177,321 keyframes; videos=873


,global_id,video_id,keyframe_id,n,pts_time,fps,frame_idx,image_path,map_mode
0,0,L21_V001,001,1,0.0000,30.0,0,/content/drive/MyDrive/data/keyframes/L21_V001...,direct_n
1,1,L21_V001,002,2,3.0000,30.0,90,/content/drive/MyDrive/data/keyframes/L21_V001...,direct_n
2,2,L21_V001,003,3,8.7000,30.0,261,/content/drive/MyDrive/data/keyframes/L21_V001...,direct_n
3,3,L21_V001,004,4,11.7333,30.0,351,/content/drive/MyDrive/data/keyframes/L21_V001...,direct_n
4,4,L21_V001,005,5,13.7000,30.0,411,/content/drive/MyDrive/data/keyframes/L21_V001...,direct_n


In [ ]:
# ============================================================
# 4. VALIDATE MASTER
# ============================================================
assert master.global_id.is_unique
assert master.image_path.map(lambda p: Path(p).exists()).all()
print('Duplicated video/keyframe:', master.duplicated(['video_id','keyframe_id']).sum())
print('Bad frame_idx:', (master.frame_idx < 0).sum())
print('Bad pts_time:', (master.pts_time < 0).sum())
print('Bad fps:', (master.fps <= 0).sum())
display(master.groupby('video_id').size().describe().to_frame('keyframes_per_video'))

Duplicated video/keyframe: 0
Bad frame_idx: 0
Bad pts_time: 0
Bad fps: 0


,keyframes_per_video
count,873.000000
mean,203.116838
std,116.080742
min,24.000000
25%,146.000000
50%,163.000000
75%,213.000000
max,632.000000


In [ ]:
master

,global_id,video_id,keyframe_id,n,pts_time,fps,frame_idx,image_path,map_mode
0,0,L21_V001,001,1,0.0000,30.0,0,/content/drive/MyDrive/data/keyframes/L21_V001...,direct_n
1,1,L21_V001,002,2,3.0000,30.0,90,/content/drive/MyDrive/data/keyframes/L21_V001...,direct_n
2,2,L21_V001,003,3,8.7000,30.0,261,/content/drive/MyDrive/data/keyframes/L21_V001...,direct_n
3,3,L21_V001,004,4,11.7333,30.0,351,/content/drive/MyDrive/data/keyframes/L21_V001...,direct_n
4,4,L21_V001,005,5,13.7000,30.0,411,/content/drive/MyDrive/data/keyframes/L21_V001...,direct_n
...,...,...,...,...,...,...,...,...,...
177316,177316,L30_V096,080,80,77.0400,25.0,1926,/content/drive/MyDrive/data/keyframes/L30_V096...,direct_n
177317,177317,L30_V096,081,81,79.6800,25.0,1992,/content/drive/MyDrive/data/keyframes/L30_V096...,direct_n
177318,177318,L30_V096,082,82,82.5200,25.0,2063,/content/drive/MyDrive/data/keyframes/L30_V096...,direct_n
177319,177319,L30_V096,083,83,87.9200,25.0,2198,/content/drive/MyDrive/data/keyframes/L30_V096...,direct_n


In [ ]:
master.to_csv('/content/drive/MyDrive/data/keyframe_master.csv')

## 5. Normalize object JSON

Baseline object JSON dùng:
- `detection_class_entities`
- `detection_boxes`
- `detection_scores`

và box dạng `[ymin, xmin, ymax, xmax]`.

Notebook lưu lại detection từ confidence `>= 0.20` (không hard-cut 0.4) và tạo summary theo keyframe để query nhanh.

In [ ]:
import pyarrow as pa
import pyarrow.parquet as pq

from datetime import (
    datetime,
    timezone
)


def normalize_existing_objects_shard(
    force=False,
    min_conf=BASE_OBJECT_MIN_CONF,
    batch_rows=100_000
):

    session_master = get_session_master(
        master,
        SESSION_ID,
        NUM_SESSIONS
    )


    # ========================================================
    # Output paths riêng cho session
    # ========================================================

    detail_path = (
        OBJECT_SHARD_DIR /
        (
            f"objects_index."
            f"part-{SESSION_ID:02d}"
            f"-of-{NUM_SESSIONS:02d}.parquet"
        )
    )

    summary_path = (
        OBJECT_SHARD_DIR /
        (
            f"keyframe_object_summary."
            f"part-{SESSION_ID:02d}"
            f"-of-{NUM_SESSIONS:02d}.parquet"
        )
    )

    done_path = (
        OBJECT_SHARD_DIR /
        (
            f"done."
            f"part-{SESSION_ID:02d}"
            f"-of-{NUM_SESSIONS:02d}.json"
        )
    )


    # ========================================================
    # Skip nếu shard đã hoàn thành
    # ========================================================

    if (
        detail_path.exists()
        and summary_path.exists()
        and done_path.exists()
        and not force
    ):

        print(
            f"Session {SESSION_ID} "
            "was already completed."
        )

        print(detail_path)
        print(summary_path)

        return


    if force:

        for p in [
            detail_path,
            summary_path,
            done_path
        ]:

            if p.exists():
                p.unlink()


    # ========================================================
    # Parquet schema
    # ========================================================

    schema = pa.schema([

        (
            "global_id",
            pa.int64()
        ),

        (
            "video_id",
            pa.string()
        ),

        (
            "keyframe_id",
            pa.string()
        ),

        (
            "label",
            pa.string()
        ),

        (
            "confidence",
            pa.float32()
        ),

        (
            "xmin_norm",
            pa.float32()
        ),

        (
            "ymin_norm",
            pa.float32()
        ),

        (
            "xmax_norm",
            pa.float32()
        ),

        (
            "ymax_norm",
            pa.float32()
        ),

        (
            "area_norm",
            pa.float32()
        ),

        (
            "center_x_norm",
            pa.float32()
        ),

        (
            "center_y_norm",
            pa.float32()
        ),
    ])


    writer = None

    buffer = []

    summary_rows = []


    found_json = 0

    missing_json = 0

    bad_json = 0

    kept_detections = 0


    # ========================================================
    # Buffered parquet writer
    # ========================================================

    def flush():

        nonlocal writer
        nonlocal buffer

        if not buffer:
            return

        table = pa.Table.from_pylist(
            buffer,
            schema=schema
        )

        if writer is None:

            writer = pq.ParquetWriter(

                detail_path,

                schema,

                compression="zstd"
            )

        writer.write_table(table)

        buffer = []


    # ========================================================
    # Process shard
    # ========================================================

    for row in tqdm(

        session_master.itertuples(
            index=False
        ),

        total=len(session_master),

        desc=(
            f"Normalize objects "
            f"session {SESSION_ID}"
        )
    ):


        # ====================================================
        # RAW object JSON
        # ====================================================

        obj_path = (

            OBJECTS_DIR /

            str(row.video_id) /

            f"{row.keyframe_id}.json"
        )


        if not obj_path.exists():

            missing_json += 1

            summary_rows.append({

                "global_id":
                    int(row.global_id),

                "video_id":
                    str(row.video_id),

                "keyframe_id":
                    str(row.keyframe_id),

                "objects_json":
                    "{}",

                "has_raw_object_json":
                    False
            })

            continue


        found_json += 1


        try:

            with open(
                obj_path,
                "r",
                encoding="utf-8"
            ) as f:

                d = json.load(f)


            labels = d.get(
                "detection_class_entities",
                []
            )

            boxes = d.get(
                "detection_boxes",
                []
            )

            scores = d.get(
                "detection_scores",
                []
            )


            if not (
                len(labels)
                == len(boxes)
                == len(scores)
            ):

                raise ValueError(
                    "Detection arrays "
                    "have different lengths."
                )


            summary = {}


            # =================================================
            # Check whether bbox coordinates are normalized
            # =================================================

            pixel_box_mode = False

            if boxes:

                flat_max = max(

                    float(x)

                    for box in boxes

                    for x in box
                )

                pixel_box_mode = (
                    flat_max > 1.5
                )


            if pixel_box_mode:

                with Image.open(
                    row.image_path
                ) as im:

                    W = im.width
                    H = im.height


            # =================================================
            # Detections
            # =================================================

            for (
                label,
                box,
                score
            ) in zip(
                labels,
                boxes,
                scores
            ):


                score = float(score)


                if score < min_conf:
                    continue


                (
                    ymin,
                    xmin,
                    ymax,
                    xmax
                ) = map(
                    float,
                    box
                )


                if pixel_box_mode:

                    xmin /= W
                    xmax /= W

                    ymin /= H
                    ymax /= H


                xmin = np.clip(
                    xmin,
                    0,
                    1
                )

                xmax = np.clip(
                    xmax,
                    0,
                    1
                )

                ymin = np.clip(
                    ymin,
                    0,
                    1
                )

                ymax = np.clip(
                    ymax,
                    0,
                    1
                )


                width = max(
                    0,
                    xmax - xmin
                )

                height = max(
                    0,
                    ymax - ymin
                )


                area = (
                    width * height
                )


                center_x = (
                    xmin + xmax
                ) / 2


                center_y = (
                    ymin + ymax
                ) / 2


                label_str = (
                    str(label).strip()
                )


                # =================================================
                # Detailed index
                # =================================================

                buffer.append({

                    "global_id":
                        int(row.global_id),

                    "video_id":
                        str(row.video_id),

                    "keyframe_id":
                        str(row.keyframe_id),

                    "label":
                        label_str,

                    "confidence":
                        np.float32(score),

                    "xmin_norm":
                        np.float32(xmin),

                    "ymin_norm":
                        np.float32(ymin),

                    "xmax_norm":
                        np.float32(xmax),

                    "ymax_norm":
                        np.float32(ymax),

                    "area_norm":
                        np.float32(area),

                    "center_x_norm":
                        np.float32(
                            center_x
                        ),

                    "center_y_norm":
                        np.float32(
                            center_y
                        ),
                })


                kept_detections += 1


                # =================================================
                # Keyframe summary
                # =================================================

                obj = summary.setdefault(

                    label_str.lower(),

                    {
                        "count": 0,

                        "max_conf":
                            0.0,

                        "max_area":
                            0.0
                    }
                )


                obj["count"] += 1


                obj["max_conf"] = max(

                    obj["max_conf"],

                    score
                )


                obj["max_area"] = max(

                    obj["max_area"],

                    area
                )


                if (
                    len(buffer)
                    >= batch_rows
                ):

                    flush()


            summary_rows.append({

                "global_id":
                    int(row.global_id),

                "video_id":
                    str(row.video_id),

                "keyframe_id":
                    str(row.keyframe_id),

                "objects_json":
                    json.dumps(
                        summary,
                        ensure_ascii=False
                    ),

                "has_raw_object_json":
                    True
            })


        except Exception as e:

            bad_json += 1


            summary_rows.append({

                "global_id":
                    int(row.global_id),

                "video_id":
                    str(row.video_id),

                "keyframe_id":
                    str(row.keyframe_id),

                "objects_json":
                    "{}",

                "has_raw_object_json":
                    True
            })


            if bad_json <= 10:

                print(
                    "[BAD JSON]",
                    obj_path,
                    e
                )


    # ========================================================
    # Final flush
    # ========================================================

    flush()


    if writer is not None:

        writer.close()


    # ========================================================
    # Summary parquet
    # ========================================================

    pd.DataFrame(
        summary_rows
    ).to_parquet(

        summary_path,

        index=False
    )


    # ========================================================
    # Completion marker
    # ========================================================

    completion = {

        "session_id":
            SESSION_ID,

        "num_sessions":
            NUM_SESSIONS,

        "processed_keyframes":
            len(session_master),

        "found_json":
            found_json,

        "missing_json":
            missing_json,

        "bad_json":
            bad_json,

        "kept_detections":
            kept_detections,

        "completed_at":
            datetime.now(
                timezone.utc
            ).isoformat()
    }


    with open(
        done_path,
        "w"
    ) as f:

        json.dump(
            completion,
            f,
            indent=2
        )


    print("\nCompleted:")
    print(json.dumps(
        completion,
        indent=2
    ))

    print("\nDetail:")
    print(detail_path)

    print("\nSummary:")
    print(summary_path)

In [ ]:
normalize_existing_objects_shard(
    force=False
)

Session 0: [0, 88,660)
Number of keyframes: 88,660
global_id: 0 -> 88659


Normalize objects session 0:   0%|          | 0/88660 [00:00<?, ?it/s]

In [ ]:
import pyarrow.dataset as ds
import pyarrow.parquet as pq


def merge_object_shards():

    detail_parts = [

        OBJECT_SHARD_DIR /
        (
            f"objects_index."
            f"part-{sid:02d}"
            f"-of-{NUM_SESSIONS:02d}.parquet"
        )

        for sid
        in range(NUM_SESSIONS)
    ]


    summary_parts = [

        OBJECT_SHARD_DIR /
        (
            f"keyframe_object_summary."
            f"part-{sid:02d}"
            f"-of-{NUM_SESSIONS:02d}.parquet"
        )

        for sid
        in range(NUM_SESSIONS)
    ]


    done_parts = [

        OBJECT_SHARD_DIR /
        (
            f"done."
            f"part-{sid:02d}"
            f"-of-{NUM_SESSIONS:02d}.json"
        )

        for sid
        in range(NUM_SESSIONS)
    ]


    # ========================================================
    # Verify all sessions completed
    # ========================================================

    missing = [

        p

        for p
        in (
            detail_parts
            + summary_parts
            + done_parts
        )

        if not p.exists()
    ]


    if missing:

        raise RuntimeError(

            "Một session chưa hoàn thành:\n"

            + "\n".join(
                str(p)
                for p in missing
            )
        )


    # ========================================================
    # Merge summary
    # ========================================================

    summary_frames = [

        pd.read_parquet(p)

        for p in summary_parts
    ]


    summary = (

        pd.concat(
            summary_frames,
            ignore_index=True
        )

        .sort_values(
            "global_id"
        )

        .reset_index(
            drop=True
        )
    )


    # ========================================================
    # Sanity checks
    # ========================================================

    assert (
        len(summary)
        == len(master)
    ), (
        f"summary={len(summary):,}, "
        f"master={len(master):,}"
    )


    assert not (
        summary[
            "global_id"
        ]
        .duplicated()
        .any()
    )


    expected = (

        master
        .sort_values(
            "global_id"
        )[
            "global_id"
        ]
        .to_numpy()
    )


    actual = (

        summary[
            "global_id"
        ]
        .to_numpy()
    )


    assert np.array_equal(
        expected,
        actual
    )


    summary.to_parquet(

        BASE_OBJECT_SUMMARY_PATH,

        index=False
    )


    # ========================================================
    # Merge detailed detections
    # ========================================================

    dataset = ds.dataset(

        [
            str(p)
            for p in detail_parts
        ],

        format="parquet"
    )


    detail_table = (
        dataset.to_table()
    )


    pq.write_table(

        detail_table,

        BASE_OBJECT_INDEX_PATH,

        compression="zstd"
    )


    print(
        "Merge completed."
    )


    print(
        "Summary:",
        BASE_OBJECT_SUMMARY_PATH
    )


    print(
        "Details:",
        BASE_OBJECT_INDEX_PATH
    )


    print(
        "Keyframes:",
        len(summary)
    )


    print(
        "Detections:",
        detail_table.num_rows
    )

merge_object_shards()

In [ ]:
# ============================================================
# 5. NORMALIZE EXISTING OBJECT JSON
# ============================================================
import pyarrow as pa
import pyarrow.parquet as pq

def image_size(path):
    with Image.open(path) as im:
        return im.width, im.height

def normalize_existing_objects(force=False, min_conf=BASE_OBJECT_MIN_CONF, batch_rows=100_000):
    if BASE_OBJECT_INDEX_PATH.exists() and BASE_OBJECT_SUMMARY_PATH.exists() and not force:
        print('Object indexes already exist.')
        return
    if BASE_OBJECT_INDEX_PATH.exists(): BASE_OBJECT_INDEX_PATH.unlink()
    if BASE_OBJECT_SUMMARY_PATH.exists(): BASE_OBJECT_SUMMARY_PATH.unlink()

    schema = pa.schema([
        ('global_id', pa.int64()), ('video_id', pa.string()), ('keyframe_id', pa.string()),
        ('label', pa.string()), ('confidence', pa.float32()),
        ('xmin_norm', pa.float32()), ('ymin_norm', pa.float32()),
        ('xmax_norm', pa.float32()), ('ymax_norm', pa.float32()),
        ('area_norm', pa.float32()), ('center_x_norm', pa.float32()), ('center_y_norm', pa.float32()),
    ])

    writer, buffer, summary_rows = None, [], []
    found = missing = bad = 0

    def flush():
        nonlocal writer, buffer
        if not buffer: return
        t = pa.Table.from_pylist(buffer, schema=schema)
        if writer is None:
            writer = pq.ParquetWriter(BASE_OBJECT_INDEX_PATH, schema, compression='zstd')
        writer.write_table(t)
        buffer = []

    for r in tqdm(master.itertuples(index=False), total=len(master), desc='Objects'):
        obj_path = OBJECTS_DIR / r.video_id / f'{r.keyframe_id}.json'
        if not obj_path.exists():
            missing += 1
            summary_rows.append({'global_id':int(r.global_id),'video_id':r.video_id,'keyframe_id':r.keyframe_id,'objects_json':'{}','has_raw_object_json':False})
            continue
        found += 1
        try:
            d = json.loads(obj_path.read_text(encoding='utf-8'))
            labels = d.get('detection_class_entities', [])
            boxes = d.get('detection_boxes', [])
            scores = d.get('detection_scores', [])
            if not (len(labels)==len(boxes)==len(scores)):
                raise ValueError('array lengths differ')

            pixel_mode = bool(boxes) and max(float(x) for b in boxes for x in b) > 1.5
            if pixel_mode: W,H = image_size(r.image_path)
            summary = {}

            for label, box, score in zip(labels, boxes, scores):
                score = float(score)
                if score < min_conf: continue
                ymin,xmin,ymax,xmax = map(float, box)
                if pixel_mode:
                    xmin,xmax = xmin/W, xmax/W
                    ymin,ymax = ymin/H, ymax/H
                xmin,ymin,xmax,ymax = [max(0.0,min(1.0,x)) for x in [xmin,ymin,xmax,ymax]]
                w,h = max(0,xmax-xmin), max(0,ymax-ymin)
                area = w*h
                label_s = str(label).strip()
                buffer.append({
                    'global_id':int(r.global_id),'video_id':r.video_id,'keyframe_id':r.keyframe_id,
                    'label':label_s,'confidence':np.float32(score),
                    'xmin_norm':np.float32(xmin),'ymin_norm':np.float32(ymin),
                    'xmax_norm':np.float32(xmax),'ymax_norm':np.float32(ymax),
                    'area_norm':np.float32(area),'center_x_norm':np.float32((xmin+xmax)/2),'center_y_norm':np.float32((ymin+ymax)/2)
                })
                item = summary.setdefault(label_s.lower(), {'count':0,'max_conf':0.0,'max_area':0.0})
                item['count'] += 1
                item['max_conf'] = max(item['max_conf'], score)
                item['max_area'] = max(item['max_area'], area)
                if len(buffer) >= batch_rows: flush()

            summary_rows.append({'global_id':int(r.global_id),'video_id':r.video_id,'keyframe_id':r.keyframe_id,'objects_json':json.dumps(summary,ensure_ascii=False),'has_raw_object_json':True})
        except Exception as e:
            bad += 1
            if bad <= 10: print('[BAD]', obj_path, e)
            summary_rows.append({'global_id':int(r.global_id),'video_id':r.video_id,'keyframe_id':r.keyframe_id,'objects_json':'{}','has_raw_object_json':True})

    flush()
    if writer is not None: writer.close()
    pd.DataFrame(summary_rows).to_parquet(BASE_OBJECT_SUMMARY_PATH, index=False)
    print(f'found={found:,}, missing={missing:,}, bad={bad:,}')
    print('summary:', BASE_OBJECT_SUMMARY_PATH)
    if BASE_OBJECT_INDEX_PATH.exists(): print('details:', BASE_OBJECT_INDEX_PATH)

normalize_existing_objects(force=False)

Objects:   0%|          | 0/177321 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# ============================================================
# 6. MEDIA-INFO INDEX
# ============================================================
def flatten_scalar_text(obj, prefix='', out=None, max_items=300):
    out = [] if out is None else out
    if len(out) >= max_items: return out
    if isinstance(obj, dict):
        for k,v in obj.items(): flatten_scalar_text(v, f'{prefix}.{k}' if prefix else str(k), out, max_items)
    elif isinstance(obj, list):
        for v in obj[:100]: flatten_scalar_text(v, prefix, out, max_items)
    elif isinstance(obj, (str,int,float,bool)) and obj is not None:
        s = str(obj).strip()
        if s and len(s) <= 3000: out.append((prefix,s))
    return out

def build_media_index(force=False):
    if MEDIA_INDEX_PATH.exists() and not force:
        return pd.read_parquet(MEDIA_INDEX_PATH)
    rows=[]
    for video_id in tqdm(sorted(master.video_id.unique()), desc='Media info'):
        cands=[MEDIA_DIR/f'{video_id}.json', MEDIA_DIR/video_id/f'{video_id}.json'] + list(MEDIA_DIR.rglob(f'{video_id}.json'))[:2]
        p=next((x for x in cands if x.exists()), None)
        text=''
        if p:
            try:
                d=json.loads(p.read_text(encoding='utf-8'))
                text=' | '.join(f'{k}: {v}' for k,v in flatten_scalar_text(d))
            except Exception: pass
        rows.append({'video_id':video_id,'metadata_text':text,'metadata_path':str(p) if p else None})
    df=pd.DataFrame(rows)
    df.to_parquet(MEDIA_INDEX_PATH,index=False)
    return df

media_df=build_media_index(force=False)
display(media_df.head())

## 7. Qwen3-VL embeddings + FAISS

Đây là bước offline lâu nhất. Có resume bằng progress JSON và tự giảm batch khi CUDA OOM.

In [ ]:
# ============================================================
# 7. LOAD QWEN3-VL EMBEDDER
# ============================================================
sys.path.insert(0, '/content/Qwen3-VL-Embedding')
from src.models.qwen3_vl_embedding import Qwen3VLEmbedder

embedder = Qwen3VLEmbedder(
    model_name_or_path=QWEN_MODEL_ID,
    max_length=4096,
    min_pixels=QWEN_MIN_PIXELS,
    max_pixels=QWEN_MAX_PIXELS,
    torch_dtype=DTYPE,
)
print('Embedder loaded.')

In [ ]:
# ============================================================
# 8. PRECOMPUTE EMBEDDINGS WITH RESUME
# ============================================================
from numpy.lib.format import open_memmap

def normalize_np(x):
    x=x.astype(np.float32,copy=False)
    n=np.maximum(np.linalg.norm(x,axis=1,keepdims=True),1e-12)
    return x/n

def build_qwen_embeddings(force=False, initial_batch_size=QWEN_BATCH_SIZE):
    N=len(master)
    if force:
        for p in [QWEN_EMBED_PATH,QWEN_PROGRESS_PATH,FAISS_PATH]:
            if Path(p).exists(): Path(p).unlink()

    if QWEN_EMBED_PATH.exists():
        emb=open_memmap(QWEN_EMBED_PATH,mode='r+')
        if emb.shape != (N,INDEX_DIM):
            raise ValueError(f'embedding shape {emb.shape} != {(N,INDEX_DIM)}')
    else:
        emb=open_memmap(QWEN_EMBED_PATH,mode='w+',dtype=np.float16,shape=(N,INDEX_DIM))

    start=0
    if QWEN_PROGRESS_PATH.exists():
        start=int(json.loads(QWEN_PROGRESS_PATH.read_text()).get('completed',0))
    print(f'Resume {start:,}/{N:,}')

    bs=initial_batch_size; i=start
    pbar=tqdm(total=N,initial=i,desc='Qwen embeddings')
    while i<N:
        cur=min(bs,N-i)
        batch=master.iloc[i:i+cur]
        inputs=[{'image':p} for p in batch.image_path.tolist()]
        try:
            with torch.inference_mode(): vec=embedder.process(inputs,normalize=True)
            vec=vec.detach().float().cpu().numpy()[:,:INDEX_DIM]
            vec=normalize_np(vec)
            emb[i:i+cur]=vec.astype(np.float16); emb.flush()
            i+=cur; pbar.update(cur)
            QWEN_PROGRESS_PATH.write_text(json.dumps({'completed':i,'total':N,'index_dim':INDEX_DIM,'model':QWEN_MODEL_ID},indent=2))
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache(); gc.collect()
            if cur<=1: raise
            bs=max(1,cur//2)
            print('OOM -> batch',bs)
    pbar.close()
    return emb

qwen_embeddings=build_qwen_embeddings(force=False)
print(qwen_embeddings.shape)

In [ ]:
# ============================================================
# 9. FAISS INDEX
# ============================================================
import faiss

def build_faiss(force=False, chunk=20_000):
    if FAISS_PATH.exists() and not force:
        idx=faiss.read_index(str(FAISS_PATH)); print('Loaded FAISS:',idx.ntotal); return idx
    emb=open_memmap(QWEN_EMBED_PATH,mode='r')
    idx=faiss.IndexFlatIP(INDEX_DIM)
    for s in tqdm(range(0,len(emb),chunk),desc='FAISS add'):
        x=np.asarray(emb[s:s+chunk],dtype=np.float32); faiss.normalize_L2(x); idx.add(x)
    faiss.write_index(idx,str(FAISS_PATH)); print('Saved',FAISS_PATH); return idx

faiss_index=build_faiss(force=False)

## 10. Query parser + multi-query retrieval

Parser local model chỉ dùng để:
- tạo English query;
- trích visually-grounded phrases;
- tìm OCR terms;
- scene/event terms.

Nếu parser lỗi JSON, fallback về query gốc.

In [ ]:
# ============================================================
# 10. LOCAL QUERY PARSER
# ============================================================
from transformers import AutoTokenizer, AutoModelForCausalLM

def parse_json_loose(text):
    m=re.search(r'\{.*\}',text,flags=re.S)
    if not m: return None
    try: return json.loads(m.group(0))
    except Exception: return None

def parse_query_local_llm(query,use_llm=True):
    fallback={'english_query':query,'grounding_phrases':[],'ocr_terms':[],'scene_terms':[]}
    if not use_llm: return fallback
    tok=AutoTokenizer.from_pretrained(PARSER_MODEL_ID)
    model=AutoModelForCausalLM.from_pretrained(PARSER_MODEL_ID,torch_dtype=DTYPE if torch.cuda.is_available() else torch.float32,device_map='auto')
    prompt=("You are a query parser for text-to-video-keyframe retrieval. Return JSON only.\n"
            'Schema:\n{"english_query":"faithful English translation","grounding_phrases":["visually detectable phrases directly stated in query"],"ocr_terms":["literal text/logo/name/number explicitly mentioned"],"scene_terms":["scene/event concepts"]}\n'
            f"Rules: do not invent objects; max 8 grounding phrases; keep useful attributes like red clothing.\nQuery: {query}")
    text=tok.apply_chat_template([{'role':'user','content':prompt}],tokenize=False,add_generation_prompt=True)
    inp=tok([text],return_tensors='pt').to(model.device)
    with torch.inference_mode(): out=model.generate(**inp,max_new_tokens=250,do_sample=False)
    generated=out[0,inp.input_ids.shape[1]:]
    parsed=parse_json_loose(tok.decode(generated,skip_special_tokens=True)) or fallback
    del model,tok,inp,out; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    for k,default in fallback.items(): parsed.setdefault(k,default)
    for k in ['grounding_phrases','ocr_terms','scene_terms']:
        parsed[k]=[str(x).strip() for x in parsed[k] if str(x).strip()][:8]
    return parsed

In [ ]:
# ============================================================
# 11. QUERY EMBEDDING + RRF
# ============================================================
from collections import defaultdict
RETRIEVAL_INSTRUCTION='Retrieve video keyframes that visually match the user event description.'

def encode_text_query(text):
    with torch.inference_mode(): v=embedder.process([{'text':text,'instruction':RETRIEVAL_INSTRUCTION}],normalize=True)
    v=v.detach().float().cpu().numpy()[:,:INDEX_DIM]
    return normalize_np(v).astype(np.float32)

def faiss_search_text(text,top_k=TOPK_PER_QUERY):
    q=encode_text_query(text); scores,ids=faiss_index.search(q,top_k)
    df=pd.DataFrame({'global_id':ids[0].astype(np.int64),'visual_score':scores[0].astype(np.float32),'rank':np.arange(1,len(ids[0])+1)})
    return df[df.global_id>=0].reset_index(drop=True)

def rrf(frames,weights=None,k=60):
    weights=weights or [1.0]*len(frames); score=defaultdict(float)
    for df,w in zip(frames,weights):
        for r in df.itertuples(index=False): score[int(r.global_id)] += float(w)/(k+int(r.rank))
    out=pd.DataFrame([{'global_id':gid,'rrf_raw':s} for gid,s in score.items()]).sort_values('rrf_raw',ascending=False)
    lo,hi=out.rrf_raw.min(),out.rrf_raw.max(); out['retrieval_score']=(out.rrf_raw-lo)/(hi-lo+1e-12)
    return out.reset_index(drop=True)

def multi_query_retrieve(query,parsed,top_k=TOPK_AFTER_RRF):
    variants=[query]
    en=str(parsed.get('english_query','')).strip()
    if en and en.lower()!=query.lower(): variants.append(en)
    concepts=parsed.get('grounding_phrases',[])+parsed.get('scene_terms',[])
    if concepts: variants.append(', '.join(concepts))
    variants=list(dict.fromkeys(v for v in variants if v.strip()))
    print('Query variants:')
    for i,v in enumerate(variants): print(f' Q{i}: {v}')
    fused=rrf([faiss_search_text(v) for v in variants]).head(top_k)
    return fused.merge(master,on='global_id',how='left'),variants

In [ ]:
# ============================================================
# 12. BASE OBJECT SCORE + METADATA BM25
# ============================================================
from rapidfuzz.fuzz import ratio
from rank_bm25 import BM25Okapi
from unidecode import unidecode

if BASE_OBJECT_SUMMARY_PATH.exists():
    object_summary_df=pd.read_parquet(BASE_OBJECT_SUMMARY_PATH)
    object_summary_map=dict(zip(object_summary_df.global_id.astype(int),object_summary_df.objects_json))
else: object_summary_map={}

def label_phrase_similarity(label,phrase):
    label,phrase=label.lower().strip(),phrase.lower().strip()
    if label==phrase: return 1.0
    if label in phrase or phrase in label: return 0.9
    return ratio(label,phrase)/100.0

def base_object_score(gid,phrases):
    if not phrases: return 0.0
    try: d=json.loads(object_summary_map.get(int(gid),'{}'))
    except Exception: d={}
    if not d: return 0.0
    vals=[]
    for phrase in phrases:
        best=0.0
        for label,info in d.items():
            sim=label_phrase_similarity(label,phrase)
            if sim<0.65: continue
            conf=float(info.get('max_conf',0)); count=int(info.get('count',1))
            count_boost=min(1.0,math.log1p(count)/math.log(6))
            best=max(best,sim*(0.8*conf+0.2*count_boost))
        if best>0: vals.append(best)
    return float(np.mean(vals)) if vals else 0.0

def tokenize_text(s): return re.findall(r'[a-z0-9_]+',unidecode(str(s).lower()))
media_corpus=[tokenize_text(x) for x in media_df.metadata_text.fillna('')]
media_bm25=BM25Okapi(media_corpus) if media_corpus else None

def metadata_scores(query):
    if media_bm25 is None: return {}
    s=np.asarray(media_bm25.get_scores(tokenize_text(query)),dtype=np.float32)
    if len(s) and s.max()>0: s=s/(s.max()+1e-12)
    return dict(zip(media_df.video_id,s.tolist()))

def add_cheap_scores(candidates,query,phrases):
    out=candidates.copy()
    out['base_object_score']=[base_object_score(g,phrases) for g in out.global_id]
    lookup=metadata_scores(query)
    out['metadata_score']=out.video_id.map(lookup).fillna(0.0).astype(float)
    out['grounding_score']=0.0; out['ocr_score']=0.0
    return out

## 13. Grounding DINO query-time verification

Chỉ chạy trên top `EXPENSIVE_VERIFY_TOPK` keyframes. Detection chi tiết được cache trong `objects/grounding-cache/`.

In [ ]:
# ============================================================
# 13. GROUNDING DINO
# ============================================================
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
_grounding_processor=None; _grounding_model=None

def load_grounding():
    global _grounding_processor,_grounding_model
    if _grounding_model is not None: return
    _grounding_processor=AutoProcessor.from_pretrained(GROUNDING_DINO_ID)
    _grounding_model=AutoModelForZeroShotObjectDetection.from_pretrained(GROUNDING_DINO_ID,torch_dtype=DTYPE if torch.cuda.is_available() else torch.float32)
    if torch.cuda.is_available(): _grounding_model=_grounding_model.to('cuda')
    _grounding_model.eval()

def unload_grounding():
    global _grounding_processor,_grounding_model
    _grounding_processor=None; _grounding_model=None; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

def grounding_hash(query,phrases,ids):
    s=json.dumps({'q':query,'p':phrases,'ids':list(map(int,ids)),'m':GROUNDING_DINO_ID},sort_keys=True)
    return hashlib.sha1(s.encode()).hexdigest()[:16]

def detection_score(phrases,labels,scores):
    vals=[]
    for phrase in phrases:
        best=0.0
        for lab,sc in zip(labels,scores):
            sim=label_phrase_similarity(str(lab),phrase)
            if sim>=0.55: best=max(best,sim*float(sc))
        vals.append(best)
    coverage=np.mean([x>0.15 for x in vals]) if vals else 0.0
    pos=[x for x in vals if x>0]; quality=float(np.mean(pos)) if pos else 0.0
    return 0.65*quality+0.35*float(coverage)

def add_grounding_scores(query,candidates,phrases,top_k=EXPENSIVE_VERIFY_TOPK,box_threshold=0.25,text_threshold=0.20):
    out=candidates.copy(); phrases=[p.strip() for p in phrases if p.strip()]
    if not phrases: return out
    top=out.head(top_k).copy(); cache=GROUNDING_CACHE_DIR/f'{grounding_hash(query,phrases,top.global_id.tolist())}.parquet'
    if cache.exists():
        c=pd.read_parquet(cache); mp=dict(zip(c.global_id.astype(int),c.grounding_score)); out['grounding_score']=out.global_id.map(mp).fillna(0.0); return out
    load_grounding(); device=next(_grounding_model.parameters()).device; rows=[]
    for r in tqdm(top.itertuples(index=False),total=len(top),desc='Grounding DINO'):
        image=Image.open(r.image_path).convert('RGB')
        inputs=_grounding_processor(images=image,text=[phrases],return_tensors='pt').to(device)
        with torch.inference_mode(): pred=_grounding_model(**inputs)
        res=_grounding_processor.post_process_grounded_object_detection(pred,threshold=box_threshold,text_threshold=text_threshold,target_sizes=[(image.height,image.width)])[0]
        scores=res['scores'].detach().float().cpu().tolist(); boxes=res['boxes'].detach().float().cpu().tolist()
        labels=[str(x) for x in res.get('text_labels',res.get('labels',[]))]
        gs=detection_score(phrases,labels,scores)
        rows.append({'global_id':int(r.global_id),'video_id':r.video_id,'keyframe_id':r.keyframe_id,'grounding_score':float(gs),'labels_json':json.dumps(labels,ensure_ascii=False),'scores_json':json.dumps(scores),'boxes_xyxy_json':json.dumps(boxes),'phrases_json':json.dumps(phrases,ensure_ascii=False)})
        del inputs,pred,res
    c=pd.DataFrame(rows); c.to_parquet(cache,index=False); mp=dict(zip(c.global_id.astype(int),c.grounding_score)); out['grounding_score']=out.global_id.map(mp).fillna(0.0)
    return out

In [ ]:
# ============================================================
# 14. LAZY OCR
# ============================================================
from rapidfuzz.fuzz import partial_ratio
_ocr_reader=None

def load_ocr():
    global _ocr_reader
    if _ocr_reader is None:
        import easyocr
        _ocr_reader=easyocr.Reader(['vi','en'],gpu=torch.cuda.is_available(),verbose=False)

def ocr_one(gid,path):
    cache=OCR_CACHE_DIR/f'{int(gid)}.json'
    if cache.exists(): return json.loads(cache.read_text(encoding='utf-8'))
    load_ocr(); result=_ocr_reader.readtext(str(path),detail=1,paragraph=False)
    rows=[{'text':str(t),'confidence':float(c),'box':np.asarray(b).tolist()} for b,t,c in result]
    cache.write_text(json.dumps(rows,ensure_ascii=False),encoding='utf-8'); return rows

def add_ocr_scores(candidates,terms,top_k=120):
    out=candidates.copy()
    if not terms: return out
    mp={}
    for r in tqdm(out.head(top_k).itertuples(index=False),total=min(top_k,len(out)),desc='OCR'):
        rows=ocr_one(r.global_id,r.image_path); text=' '.join(x['text'] for x in rows)
        mp[int(r.global_id)]=float(np.mean([partial_ratio(unidecode(t.lower()),unidecode(text.lower()))/100 for t in terms])) if text else 0.0
    out['ocr_score']=out.global_id.map(mp).fillna(0.0); return out

In [ ]:
# ============================================================
# 15. FRAME FUSION + TEMPORAL GROUPING
# ============================================================
PRE_RERANK_WEIGHTS={'retrieval_score':0.60,'base_object_score':0.10,'grounding_score':0.18,'metadata_score':0.07,'ocr_score':0.05}

def compute_pre_score(df):
    out=df.copy()
    for c in PRE_RERANK_WEIGHTS:
        if c not in out: out[c]=0.0
    out['pre_rerank_score']=sum(w*out[c].astype(float) for c,w in PRE_RERANK_WEIGHTS.items())
    return out.sort_values('pre_rerank_score',ascending=False).reset_index(drop=True)

def temporal_groups(candidates,gap=TEMPORAL_GAP_SECONDS):
    c=candidates.sort_values(['video_id','pts_time']).copy(); rows=[]; sid=0
    for video_id,g in c.groupby('video_id',sort=False):
        current=[]; prev=None
        def flush(frames):
            nonlocal sid
            if not frames: return
            seg=pd.DataFrame(frames); s=np.sort(seg.pre_rerank_score.to_numpy(float))[::-1]
            score=0.55*float(s[0])+0.35*float(s[:3].mean())+0.10*min(1.0,len(seg)/5)
            rep=seg.sort_values('pre_rerank_score',ascending=False).iloc[0]
            rows.append({'segment_id':sid,'video_id':video_id,'start_time':float(seg.pts_time.min()),'end_time':float(seg.pts_time.max()),'n_candidates':len(seg),'segment_score':score,'representative_global_id':int(rep.global_id),'representative_keyframe_id':rep.keyframe_id,'representative_frame_idx':int(rep.frame_idx),'member_global_ids_json':json.dumps(seg.global_id.astype(int).tolist())}); sid+=1
        for r in g.itertuples(index=False):
            if prev is None or float(r.pts_time)-prev<=gap: current.append(r._asdict())
            else: flush(current); current=[r._asdict()]
            prev=float(r.pts_time)
        flush(current)
    out=pd.DataFrame(rows)
    return out.sort_values('segment_score',ascending=False).reset_index(drop=True) if len(out) else out

def row_from_gid(gid): return master.loc[master.global_id==int(gid)].iloc[0]

def neighbor_rows(gid,radius=2):
    r=row_from_gid(gid); vg=master[master.video_id==r.video_id].sort_values('pts_time').reset_index(drop=True); pos=vg.index[vg.global_id==int(gid)].tolist()[0]
    return vg.iloc[max(0,pos-radius):min(len(vg),pos+radius+1)].copy()

def neighbor_images(gid,radius=1): return neighbor_rows(gid,radius).image_path.tolist()

In [ ]:
# ============================================================
# 16. QWEN3-VL RERANKER
# ============================================================
from src.models.qwen3_vl_reranker import Qwen3VLReranker
_reranker=None

def load_reranker():
    global _reranker
    if _reranker is None:
        _reranker=Qwen3VLReranker(model_name_or_path=QWEN_RERANKER_ID,max_length=4096,min_pixels=QWEN_MIN_PIXELS,max_pixels=QWEN_MAX_PIXELS,torch_dtype=DTYPE)

def unload_reranker():
    global _reranker
    _reranker=None; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

def rerank_segments(query,segments,top_k=RERANK_TOP_SEGMENTS):
    if not len(segments): return segments
    load_reranker(); top=segments.head(top_k).copy(); docs=[]
    for r in top.itertuples(index=False):
        docs.append({'image':neighbor_images(r.representative_global_id,1),'text':f'Video segment around {r.start_time:.2f}-{r.end_time:.2f} seconds.'})
    inp={'instruction':'Rank video-keyframe segments by how completely their visual content matches the textual event description.','query':{'text':query},'documents':docs}
    with torch.inference_mode(): scores=_reranker.process(inp)
    top['reranker_score']=np.asarray(scores,dtype=np.float32); top['final_segment_score']=0.45*top.segment_score.astype(float)+0.55*top.reranker_score.astype(float)
    rest=segments.iloc[top_k:].copy()
    if len(rest):
        rest['reranker_score']=0.0; rest['final_segment_score']=0.45*rest.segment_score.astype(float); top=pd.concat([top,rest],ignore_index=True)
    return top.sort_values('final_segment_score',ascending=False).reset_index(drop=True)

In [ ]:
# ============================================================
# 17. DIVERSITY-AWARE TOP-100
# ============================================================
def build_top100(segments,max_results=100):
    if not len(segments): return pd.DataFrame()
    gids=[]; seen=set()
    def add(g):
        g=int(g)
        if g not in seen and len(gids)<max_results: seen.add(g); gids.append(g)

    # Top-3 representatives first
    for r in segments.head(3).itertuples(index=False): add(r.representative_global_id)

    # Temporal hedge top-1: previous + next keyframe
    topgid=int(segments.iloc[0].representative_global_id); center=float(row_from_gid(topgid).pts_time); ng=neighbor_rows(topgid,1)
    prev=ng[ng.pts_time<center].sort_values('pts_time',ascending=False); nxt=ng[ng.pts_time>center].sort_values('pts_time')
    if len(prev): add(prev.iloc[0].global_id)
    if len(nxt): add(nxt.iloc[0].global_id)

    # Remaining representatives
    for r in segments.iloc[3:].itertuples(index=False):
        add(r.representative_global_id)
        if len(gids)>=max_results: break

    # Fill with neighbors of strongest segments
    if len(gids)<max_results:
        for r in segments.itertuples(index=False):
            center=float(row_from_gid(r.representative_global_id).pts_time); ng=neighbor_rows(r.representative_global_id,3).copy(); ng['dist']=(ng.pts_time-center).abs(); ng=ng.sort_values('dist')
            for g in ng.global_id.tolist():
                add(g)
                if len(gids)>=max_results: break
            if len(gids)>=max_results: break

    rows=[]
    for rank,g in enumerate(gids,1):
        r=row_from_gid(g); rows.append({'rank':rank,'video_id':r.video_id,'frame_idx':int(r.frame_idx),'keyframe_id':r.keyframe_id,'pts_time':float(r.pts_time),'global_id':int(g),'image_path':r.image_path})
    return pd.DataFrame(rows)

In [ ]:
# ============================================================
# 18. END-TO-END SEARCH
# ============================================================
def search_textual_kis(query,use_query_parser=True,use_grounding=True,use_ocr=True,use_reranker=True,return_intermediate=False):
    print('='*80); print('QUERY:',query); print('='*80)
    parsed=parse_query_local_llm(query,use_llm=use_query_parser)
    print(json.dumps(parsed,ensure_ascii=False,indent=2))

    candidates,variants=multi_query_retrieve(query,parsed)
    candidates=add_cheap_scores(candidates,query,parsed.get('grounding_phrases',[]))

    if use_grounding and parsed.get('grounding_phrases'):
        candidates=add_grounding_scores(query,candidates,parsed['grounding_phrases'])
    if use_ocr and parsed.get('ocr_terms'):
        candidates=add_ocr_scores(candidates,parsed['ocr_terms'])

    candidates=compute_pre_score(candidates)
    segments=temporal_groups(candidates.head(700))
    print('candidate frames:',len(candidates),'segments:',len(segments))

    if use_reranker: segments=rerank_segments(query,segments)
    else:
        segments['reranker_score']=0.0; segments['final_segment_score']=segments['segment_score']

    submission=build_top100(segments,100)
    display(submission.head(10))
    if return_intermediate: return {'parsed':parsed,'variants':variants,'candidates':candidates,'segments':segments,'submission':submission}
    return submission

In [ ]:
# ============================================================
# 19. RUN ONE QUERY
# ============================================================
QUERY='Tìm video về một diễn giả mặc áo đỏ phát biểu tại một cuộc họp báo ngoài trời, phía sau có nhiều cây xanh.'
result=search_textual_kis(QUERY,use_query_parser=True,use_grounding=True,use_ocr=True,use_reranker=True,return_intermediate=True)
top100=result['submission']

In [ ]:
# ============================================================
# 20. VISUALIZE TOP RESULTS
# ============================================================
import matplotlib.pyplot as plt

def show_top(df,n=12,cols=4):
    if not len(df): return
    n=min(n,len(df)); rows=math.ceil(n/cols); fig,axes=plt.subplots(rows,cols,figsize=(16,4*rows)); axes=np.atleast_1d(axes).ravel()
    for ax in axes: ax.axis('off')
    for i,r in enumerate(df.head(n).itertuples(index=False)):
        axes[i].imshow(Image.open(r.image_path).convert('RGB')); axes[i].set_title(f'#{r.rank} {r.video_id}\nkf={r.keyframe_id} frame={r.frame_idx} t={r.pts_time:.2f}s'); axes[i].axis('off')
    plt.tight_layout(); plt.show()
show_top(top100)

In [ ]:
# ============================================================
# 21. EXPORT TOP-100
# ============================================================
OUTPUT_CSV=ARTIFACT_DIR/'textual_kis_top100.csv'
top100[['video_id','frame_idx']].to_csv(OUTPUT_CSV,index=False,header=False)
print('Saved:',OUTPUT_CSV)

In [ ]:
# ============================================================
# 22. TEXTUAL KIS METRIC HELPER
# ============================================================
def evaluate_textual_kis_submission(df,gt_video,gt_start_frame,gt_end_frame):
    hit=[int(str(r.video_id)==str(gt_video) and int(gt_start_frame)<=int(r.frame_idx)<=int(gt_end_frame)) for r in df.itertuples(index=False)]
    metrics={}
    for k in [1,5,20,50,100]: metrics[f'R@{k}']=max(hit[:k],default=0)
    metrics['FinalScore']=float(np.mean([metrics[f'R@{k}'] for k in [1,5,20,50,100]]))
    return metrics

# Example:
# evaluate_textual_kis_submission(top100, 'L01_V001', 500, 510)

# Tuning / ablation order

Sau khi chạy end-to-end, nên đo validation theo thứ tự:

1. Qwen retrieval only.
2. + multi-query RRF.
3. + Faster R-CNN object evidence.
4. + Grounding DINO.
5. + metadata.
6. + temporal grouping.
7. + Qwen3-VL-Reranker.
8. + lazy OCR.
9. Tune `PRE_RERANK_WEIGHTS`, `TEMPORAL_GAP_SECONDS`, `RERANK_TOP_SEGMENTS`.
10. Tune chiến lược Top-100 diversity/temporal hedge trực tiếp theo `R@1, R@5, R@20, R@50, R@100`.

Artifacts chính:

```text
data/
├── objects/
│   ├── objects_index.parquet
│   ├── keyframe_object_summary.parquet
│   └── grounding-cache/
└── retrieval-artifacts/
    ├── master_keyframes.parquet
    ├── media_index.parquet
    ├── qwen3_vl_embedding_1024d.npy
    ├── qwen3_vl_1024d.index
    ├── ocr-cache/
    └── textual_kis_top100.csv
```

Nếu CUDA OOM:
- giảm `QWEN_BATCH_SIZE`;
- giảm `QWEN_MAX_PIXELS` từ `768*768` xuống `512*512`;
- giảm `RERANK_TOP_SEGMENTS`;
- sau Grounding DINO gọi `unload_grounding()`;
- sau reranking gọi `unload_reranker()`.